# TinyLlama Roleplay SLM Training Pipeline

This notebook provides a clean, modular, and reproducible LoRA fine-tuning pipeline for roleplay-focused conversational training on Google Colab GPUs.

## 1. Dependency installation

In [ ]:
# Core training stack (pinned for Colab stability)
!pip -q install -U   "transformers==4.41.2"   "datasets==2.20.0"   "accelerate==0.31.0"   "peft==0.11.1"   "trl==0.9.4"   "bitsandbytes==0.43.1"   "sentencepiece"   "evaluate"


## 2. Imports

In [ ]:
import os
import re
import random
import unicodedata
from typing import Dict, List, Optional, Tuple

import torch
from datasets import Dataset, concatenate_datasets, load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    set_seed,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig


## 3. GPU detection

In [ ]:
def print_gpu_info() -> Dict[str, str]:
    info = {
        "cuda_available": str(torch.cuda.is_available()),
        "device_count": str(torch.cuda.device_count()) if torch.cuda.is_available() else "0",
    }
    if torch.cuda.is_available():
        info["gpu_name"] = torch.cuda.get_device_name(0)
        info["total_memory_gb"] = f"{torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f}"
        info["bf16_supported"] = str(torch.cuda.is_bf16_supported())
    for k, v in info.items():
        print(f"{k}: {v}")
    return info

gpu_info = print_gpu_info()
SEED = 42
set_seed(SEED)
random.seed(SEED)


## 4. Model and tokenizer loading

In [ ]:
MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
MAX_LENGTH = 1024

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model.config.use_cache = False


## 5. LoRA configuration

In [ ]:
lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


## 6. Dataset loading

In [ ]:
DATASET_SOURCES = {
    "roleplay": [
        ("AlekseyKorshuk/persona-chat", "train"),
        ("blended_skill_talk", "train"),
    ],
    "multi_turn": [
        ("HuggingFaceH4/ultrachat_200k", "train_sft"),
    ],
    "instruction": [
        ("yahma/alpaca-cleaned", "train"),
    ],
    "personality": [
        ("fka/awesome-chatgpt-prompts", "train"),
    ],
}

def safe_load(name: str, split: str):
    try:
        ds = load_dataset(name, split=split)
        print(f"Loaded {name}:{split} -> {len(ds):,} rows")
        return ds
    except Exception as exc:
        print(f"Failed {name}:{split}: {exc}")
        return None

loaded_sources = {
    category: [safe_load(name, split) for name, split in specs]
    for category, specs in DATASET_SOURCES.items()
}


## 7. Dataset cleaning

In [ ]:
HTML_RE = re.compile(r"<[^>]+>")
SYMBOL_RE = re.compile(r"[^\w\s\.,!?;:'\"()\-\n]")
REPEAT_TOKEN_RE = re.compile(r"\b(\w+)(\s+\1\b){2,}", flags=re.IGNORECASE)
MULTISPACE_RE = re.compile(r"\s+")

def normalize_text(text: str) -> str:
    text = unicodedata.normalize("NFKC", str(text))
    text = HTML_RE.sub(" ", text)
    text = SYMBOL_RE.sub(" ", text)
    text = REPEAT_TOKEN_RE.sub(r"\1", text)
    text = text.replace("…", "...")
    text = MULTISPACE_RE.sub(" ", text).strip()
    return text

def is_valid_turn(user: str, assistant: str) -> bool:
    if not user or not assistant:
        return False
    if len(assistant) < 20:
        return False
    if re.fullmatch(r"[\W_]+", user + assistant):
        return False
    return True


## 8. Dataset format normalization

In [ ]:
SYSTEM_PREFIX = "You are an immersive roleplay assistant. Stay in character, emotionally consistent, descriptive, and coherent across turns."

def to_chat_text(system_prompt: str, turns: List[Tuple[str, str]]) -> str:
    chunks = [f"<|system|>\n{normalize_text(system_prompt or SYSTEM_PREFIX)}\n"]
    for user, assistant in turns:
        u, a = normalize_text(user), normalize_text(assistant)
        if is_valid_turn(u, a):
            chunks.append(f"<|user|>\n{u}\n")
            chunks.append(f"<|assistant|>\n{a}\n")
    return "\n".join(chunks).strip()

def extract_turns(example: Dict, category: str) -> Optional[Dict]:
    turns = []
    system_prompt = example.get("system", SYSTEM_PREFIX)

    if "conversations" in example and isinstance(example["conversations"], list):
        convo = example["conversations"]
        pending_user = None
        for msg in convo:
            role = str(msg.get("from") or msg.get("role") or "").lower()
            content = msg.get("value") or msg.get("content") or ""
            if role in {"human", "user"}:
                pending_user = content
            elif role in {"assistant", "gpt"} and pending_user is not None:
                turns.append((pending_user, content))
                pending_user = None
    elif {"instruction", "output"}.issubset(example.keys()):
        instruction = example.get("instruction", "")
        inp = example.get("input", "")
        user = f"{instruction}\n{inp}".strip()
        turns = [(user, example.get("output", ""))]
    elif {"act", "prompt"}.issubset(example.keys()):
        system_prompt = f"Roleplay as: {example.get('act', '')}"
        turns = [("Introduce yourself in-character.", example.get("prompt", ""))]
    elif {"text"}.issubset(example.keys()):
        txt = str(example.get("text", ""))
        if "<|user|>" in txt and "<|assistant|>" in txt:
            return {"text": normalize_text(txt), "category": category, "turn_count": txt.count("<|assistant|>")}

    if not turns:
        return None

    chat = to_chat_text(system_prompt, turns)
    if "<|assistant|>" not in chat:
        return None

    return {"text": chat, "category": category, "turn_count": len(turns)}


## 9. Dataset merging and balancing

In [ ]:
TARGET_MIX = {
    "roleplay": 0.50,
    "multi_turn": 0.25,
    "instruction": 0.15,
    "personality": 0.10,
}
TARGET_SIZE = 1_000_000  # Increase up to 3M if storage/runtime permits.

def normalize_category_dataset(ds, category: str) -> Dataset:
    rows = []
    for ex in ds:
        parsed = extract_turns(ex, category)
        if not parsed:
            continue
        if len(parsed["text"]) <= 40:
            continue
        rows.append(parsed)
    normalized = Dataset.from_list(rows)
    normalized = normalized.drop_duplicates("text")
    return normalized

category_sets = {}
for category, sources in loaded_sources.items():
    normalized_parts = []
    for ds in sources:
        if ds is None:
            continue
        normalized_parts.append(normalize_category_dataset(ds, category))
    if normalized_parts:
        merged = concatenate_datasets(normalized_parts) if len(normalized_parts) > 1 else normalized_parts[0]
        merged = merged.drop_duplicates("text")
        category_sets[category] = merged
        print(f"{category}: {len(merged):,}")

if not category_sets:
    raise RuntimeError("No datasets were loaded successfully. Check internet access and dataset names.")

available = [
    int(len(category_sets[c]) / TARGET_MIX[c])
    for c in TARGET_MIX
    if c in category_sets and len(category_sets[c]) > 0
]
if not available:
    raise RuntimeError("Loaded datasets are empty after normalization.")

max_size_by_pool = min(available)
final_target_size = max(100_000, min(TARGET_SIZE, max_size_by_pool))
print(f"Final target size: {final_target_size:,}")

balanced_parts = []
for category, ratio in TARGET_MIX.items():
    if category not in category_sets:
        continue
    n = int(final_target_size * ratio)
    pool = category_sets[category].shuffle(seed=SEED)
    take = min(n, len(pool))
    balanced_parts.append(pool.select(range(take)))
    print(f"Selected {take:,} from {category}")

full_dataset = concatenate_datasets(balanced_parts).shuffle(seed=SEED)
print(full_dataset)


## 10. Tokenization

In [ ]:
def tokenize_batch(batch: Dict[str, List[str]]) -> Dict[str, List[List[int]]]:
    tokenized = tokenizer(
        batch["text"],
        max_length=MAX_LENGTH,
        truncation=True,
        padding="max_length",
    )
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

tokenized_dataset = full_dataset.map(
    tokenize_batch,
    batched=True,
    batch_size=512,
    num_proc=2,
    remove_columns=full_dataset.column_names,
    desc="Tokenizing",
)


## 11. Train / validation split

In [ ]:
split_dataset = tokenized_dataset.train_test_split(test_size=0.10, seed=SEED)
train_dataset = split_dataset["train"]
val_dataset = split_dataset["test"]
print(f"Train: {len(train_dataset):,} | Validation: {len(val_dataset):,}")


## 12. Training configuration

In [ ]:
OUTPUT_DIR = "/content/tinyllama-roleplay-lora"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    num_train_epochs=3,
    evaluation_strategy="steps",
    eval_steps=500,
    save_steps=500,
    save_total_limit=2,
    logging_steps=50,
    fp16=True,
    gradient_checkpointing=True,
    max_grad_norm=1.0,
    report_to="none",
)

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    packing=False,
    max_seq_length=MAX_LENGTH,
    dataset_text_field="text",
)


## 13. Training loop

In [ ]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    args=training_args,
)

train_result = trainer.train()
trainer.save_state()


## 14. Evaluation

In [ ]:
eval_metrics = trainer.evaluate()
print(eval_metrics)

def generate_reply(prompt: str, max_new_tokens: int = 180) -> str:
    model.eval()
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.8,
            top_p=0.9,
            repetition_penalty=1.08,
            do_sample=True,
            eos_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(output_ids[0], skip_special_tokens=False)

def simple_roleplay_scores(text: str, expected_markers: List[str]) -> Dict[str, float]:
    coherence = 1.0 if len(text) > 80 else 0.0
    consistency = sum(marker.lower() in text.lower() for marker in expected_markers) / max(1, len(expected_markers))
    length_score = min(1.0, len(text) / 400)
    return {
        "coherence": coherence,
        "character_consistency": consistency,
        "response_length": length_score,
    }

roleplay_prompts = [
    "<|system|>\nYou are a medieval knight speaking to a traveler.\n<|user|>\nWhat dangers await us on this road?\n<|assistant|>",
    "<|system|>\nAct as an AI companion living on Mars.\n<|user|>\nHow was your day in the colony?\n<|assistant|>",
]

for p in roleplay_prompts:
    out = generate_reply(p)
    scores = simple_roleplay_scores(out, ["knight", "Mars", "traveler", "colony"])
    print("\nPROMPT:\n", p)
    print("\nOUTPUT:\n", out)
    print("\nSCORES:\n", scores)


## 15. Model export

In [ ]:
ADAPTER_DIR = "/content/tinyllama-roleplay-adapter"
MERGED_DIR = "/content/tinyllama-roleplay-merged"
TOKENIZER_DIR = "/content/tinyllama-roleplay-tokenizer"

trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(TOKENIZER_DIR)

merged_model = trainer.model.merge_and_unload()
merged_model.save_pretrained(MERGED_DIR, safe_serialization=True, max_shard_size="2GB")

print("Saved adapter:", ADAPTER_DIR)
print("Saved merged model:", MERGED_DIR)
print("Saved tokenizer:", TOKENIZER_DIR)
